# MASTER PIPELINE OUTPUT 3

## Plantation Condition & Inspection Priority

**AOI:** SIAK_AOI01  
**Reference year:** 2025  
**Architecture:** Python/Jupyter + Google Earth Engine backend  
**Analytical status:** FROZEN  
**Field validation:** NOT AVAILABLE

This notebook is the official reproducibility master for
Output 3 of the SDAI Palm Oil Monitoring pilot.

### Workflow represented

`30A -> 30B -> 30C -> 30D -> 30E -> 30F -> 30G -> 30H -> 30I`

### Important architecture rule

The hybrid GEE + Python route does **not** duplicate raw
satellite preprocessing locally.

Google Earth Engine remains the source of truth for:

- source acquisition;
- cloud / quality masking;
- Sentinel-1 / Sentinel-2 composites;
- matched temporal baseline;
- anomaly products;
- frozen core risk;
- contextual raster generation.

Python/Jupyter is used for:

- controlled asset access;
- sampling / analysis tables;
- feature screening;
- redundancy / VIF analysis;
- spatial/group validation;
- context calibration;
- frozen integration;
- QA;
- documentation;
- packaging and export orchestration.

Large persistent raster assets remain in Earth Engine.

### Scientific interpretation

This is an **inspection-priority / early-warning product**.

It is **not**:

- a confirmed disease diagnosis;
- a nutrient-deficiency diagnosis;
- a field-validated plantation-health map.

Resampling does not create new spatial information.
Weather, hydrology and soil context retain their original
effective information scale.

## 0. Execution Policy

The default master mode is conservative:

- read persistent assets;
- restore frozen parameters;
- reproduce / audit the final product;
- do not automatically start exports;
- do not automatically retune AOI01.

For a new AOI, use the reusable AOI runner produced in 30J-C.

In [ ]:
from pathlib import Path
import json
import ssl
import certifi
import ee
import pandas as pd
import numpy as np

START_EXPORTS = False
ALLOW_MODEL_RETUNING = False


def find_project_root():

    start = Path.cwd().resolve()

    candidates = [
        start,
        *start.parents
    ]

    for candidate in candidates:

        if (
            (candidate / "OUTPUT3_COMPLETE_PACKAGE").exists()
            and
            (candidate / "checkpoints_30F").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Output 3 project root."
    )


PROJECT_ROOT = find_project_root()

PACKAGE_ROOT = (
    PROJECT_ROOT /
    "OUTPUT3_COMPLETE_PACKAGE"
)

print("Project root :", PROJECT_ROOT)
print("Package root :", PACKAGE_ROOT)
print("START_EXPORTS :", START_EXPORTS)
print("ALLOW_MODEL_RETUNING :", ALLOW_MODEL_RETUNING)

## 1. AOI Configuration

AOI-specific values are separated from processing logic.

This AOI01 configuration is the frozen reference used to
reproduce the current pilot.

In [ ]:
CONFIG_PATH = (
    PACKAGE_ROOT /
    "01_CONFIG" /
    "SIAK_AOI01_OUTPUT3_CONFIG.json"
)

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    cfg = json.load(f)

print("AOI ID         :", cfg["aoi_id"])
print("Reference year :", cfg["reference_year"])
print("EE project     :", cfg["project_id"])

assert cfg["aoi_id"] == "SIAK_AOI01"
assert cfg["reference_year"] == 2025

## 2. Earth Engine Initialization

Earth Engine performs the wall-to-wall raster operations
server-side. Python is the orchestration and analytical
environment.

In [ ]:
# Apply the known Windows/certifi session fix.
ssl._create_default_https_context = (
    lambda:
    ssl.create_default_context(
        cafile=certifi.where()
    )
)

try:

    ee.Initialize(
        project=cfg["project_id"]
    )

except Exception:

    print(
        "Earth Engine initialization failed. "
        "Authenticate only if required."
    )

    ee.Authenticate()

    ee.Initialize(
        project=cfg["project_id"]
    )

print(
    "Earth Engine initialization : PASS"
)

## 3. Complete 30A-30I Process Contract

The master preserves the full workflow history even though
the hybrid route begins Python analytics from persistent
30B-30E Earth Engine assets.

In [ ]:
STAGE_CATALOG = (
    PACKAGE_ROOT /
    "09_DOCUMENTATION" /
    "OUTPUT3_PROCESS_STAGE_CATALOG.csv"
)

stage_df = pd.read_csv(
    STAGE_CATALOG
)

display(stage_df)

# 30A-30E | Persistent Geospatial Source Pipeline

These stages constitute the upstream Earth Engine processing
contract.

### 30A
AOI, upstream data availability and source contract.

### 30B
Matched baseline and monitoring optical/radar composites.

### 30C
Period anomaly products.

### 30D
Core condition / persistence / inspection-risk logic.

The frozen core used downstream is:

`SIAK_AOI01_O3_30D04_BALANCED_RISK_2025`

### 30E
Contextual enrichment:

- 30E01 Tree age/event context;
- 30E02A CHIRPS rainfall;
- 30E02B ERA5-Land;
- 30E02C ET/PET;
- 30E03A terrain;
- 30E03B hydrology;
- 30E04A soil profile;
- 30E04B soil-water retention;
- 30E05 GEDI supporting QA only.

GEDI is not interpolated wall-to-wall.

In [ ]:
ASSET_INVENTORY_PATH = (
    PACKAGE_ROOT /
    "02_ASSET_INVENTORY" /
    "OUTPUT3_GEE_ASSET_INVENTORY.csv"
)

asset_df = pd.read_csv(
    ASSET_INVENTORY_PATH
)

required_stages = [
    "30B",
    "30C",
    "30D",
    "30E01",
    "30E02",
    "30E03",
    "30E04",
    "30G",
    "UPSTREAM_O1",
    "UPSTREAM_O2",
]

stage_counts = (
    asset_df["stage"]
    .value_counts()
)

print(
    "Persistent assets by stage:"
)

display(
    stage_counts
    .rename("asset_count")
    .to_frame()
)

missing_stages = [
    stage
    for stage in required_stages
    if stage not in stage_counts.index
]

if missing_stages:

    raise RuntimeError(
        f"Missing required asset stages: {missing_stages}"
    )

print(
    "30A-30E persistent source contract : PASS"
)

In [ ]:
O3 = cfg["output3_assets"]

ASSETS = {

    "CORE":
        O3["core"],

    "AGE":
        O3["age"],

    "CHIRPS":
        O3["chirps"],

    "ERA5":
        O3["era5_land"],

    "ETPET":
        O3["et_pet"],

    "TERRAIN":
        O3["terrain"],

    "HYDRO":
        O3["hydrology"],

    "SOIL":
        O3["soil"],

    "SOIL_WATER":
        O3["soil_water"],

    "CANONICAL":
        O3["canonical_final"],
}


for key, asset_id in ASSETS.items():

    info = ee.data.getAsset(
        asset_id
    )

    print(
        f"{key:12s}: PASS | "
        f"{asset_id.split('/')[-1]}"
    )

# 30F | Python Feature Screening & Frozen Calibration

30F is the analytical bridge between persistent GEE context
assets and the final integrated product.

The historical experimental notebook is preserved separately
in the package archive.

The final production reconstruction uses only the **frozen**
30F calibration and family weights.

This prevents accidental retuning during routine reruns.

In [ ]:
CHECKPOINT_30F = (
    PROJECT_ROOT /
    "checkpoints_30F"
)

CALIBRATION_PATH = (
    CHECKPOINT_30F /
    "30F_B18C_AOI01_WALL_TO_WALL_CALIBRATION_V1.csv"
)

FAMILY_WEIGHT_PATH = (
    CHECKPOINT_30F /
    "30F_B18C_FAMILY_WEIGHTS_V1.csv"
)

if not CALIBRATION_PATH.exists():
    raise FileNotFoundError(
        CALIBRATION_PATH
    )

if not FAMILY_WEIGHT_PATH.exists():
    raise FileNotFoundError(
        FAMILY_WEIGHT_PATH
    )


calibration_df = pd.read_csv(
    CALIBRATION_PATH
)

family_weight_df = pd.read_csv(
    FAMILY_WEIGHT_PATH
)


if len(calibration_df) != 8:
    raise ValueError(
        "Expected 8 frozen modifier features."
    )

if len(family_weight_df) != 4:
    raise ValueError(
        "Expected 4 frozen context families."
    )


print(
    "Frozen modifier features :",
    len(calibration_df)
)

print(
    "Frozen families          :",
    len(family_weight_df)
)

display(calibration_df)

display(family_weight_df)

# 30G | Frozen Wall-to-Wall Integrated Product

This section reconstructs the final integration using:

- unchanged B18C calibration;
- frozen feature directions;
- frozen within-family weights;
- frozen family weights;
- F05 modifier amplitude;
- evidence-gated class transitions;
- maximum class change of ±1.

No fitting or tuning occurs here.

In [ ]:
# ------------------------------------------------------------
# Feature source registry
# ------------------------------------------------------------

feature_source_registry = {

    "LOCAL_RELIEF_300M_M":
        ASSETS["TERRAIN"],

    "SLOPE_DEG":
        ASSETS["TERRAIN"],

    "HAND_M":
        ASSETS["HYDRO"],

    "SILT_PCT_DEEP_100_200CM":
        ASSETS["SOIL"],

    "SOC_GKG_SURFACE_0_30CM":
        ASSETS["SOIL"],

    "WV1500_DEEP_100_200CM":
        ASSETS["SOIL_WATER"],

    "WV10_DEEP_100_200CM":
        ASSETS["SOIL_WATER"],

    "PAW_PROXY_SURFACE_0_30CM":
        ASSETS["SOIL_WATER"],
}


# ------------------------------------------------------------
# Core
# ------------------------------------------------------------

core_img = ee.Image(
    ASSETS["CORE"]
)

core_class = (
    core_img
    .select("CONDITION_CLASS")
    .round()
    .toInt16()
)

core_raw = (
    core_img
    .select("CONDITION_CLASS_RAW")
    .round()
    .toInt16()
)

core_risk = (
    core_img
    .select("RISK_SCORE_0_100")
    .toFloat()
)

valid_mask = (
    core_class.gte(1)
    .And(
        core_class.lte(4)
    )
)


# ------------------------------------------------------------
# Directional feature normalization
#
# P10 -> 0
# P90 -> 1
# negative associations inverted
# [0,1] -> [-1,+1]
# ------------------------------------------------------------

directional_images = {}

for _, row in calibration_df.iterrows():

    feature = row["feature"]

    source = (
        ee.Image(
            feature_source_registry[
                feature
            ]
        )
        .select(feature)
        .toFloat()
        .updateMask(valid_mask)
    )

    p10 = float(
        row["p10"]
    )

    p90 = float(
        row["p90"]
    )

    direction = int(
        row["association_direction"]
    )

    norm01 = (
        source
        .subtract(p10)
        .divide(
            p90 - p10
        )
        .clamp(
            0,
            1
        )
    )

    if direction < 0:

        norm01 = (
            ee.Image.constant(1)
            .subtract(norm01)
        )

    directional_images[
        feature
    ] = (
        norm01
        .multiply(2)
        .subtract(1)
        .rename(
            f"DIR_{feature}"
        )
    )


# ------------------------------------------------------------
# Within-family composites
# ------------------------------------------------------------

family_score_images = {}
family_coverage_images = {}

for family in (
    family_weight_df[
        "family"
    ].tolist()
):

    family_cal = (
        calibration_df[
            calibration_df[
                "family"
            ] == family
        ]
    )

    weighted_sum = (
        ee.Image.constant(0)
        .toFloat()
    )

    available_weight = (
        ee.Image.constant(0)
        .toFloat()
    )

    for _, row in family_cal.iterrows():

        feature = row["feature"]

        weight = float(
            row[
                "within_family_weight"
            ]
        )

        img = directional_images[
            feature
        ]

        valid = (
            img.mask()
            .reduce(
                ee.Reducer.min()
            )
            .gt(0)
        )

        weighted_sum = (
            weighted_sum
            .add(
                img
                .unmask(0)
                .multiply(weight)
            )
        )

        available_weight = (
            available_weight
            .add(
                valid
                .toFloat()
                .multiply(weight)
            )
        )

    family_score = (
        weighted_sum
        .divide(
            available_weight
            .max(1e-6)
        )
        .updateMask(
            available_weight.gte(0.50)
        )
        .clamp(
            -1,
            1
        )
        .rename(
            f"FAMILY_SCORE_{family}"
        )
    )

    family_score_images[
        family
    ] = family_score

    family_coverage_images[
        family
    ] = (
        available_weight
        .rename(
            f"FAMILY_COVERAGE_{family}"
        )
    )


# ------------------------------------------------------------
# Family-balanced overall context modifier
# ------------------------------------------------------------

overall_sum = (
    ee.Image.constant(0)
    .toFloat()
)

overall_weight = (
    ee.Image.constant(0)
    .toFloat()
)

for _, row in family_weight_df.iterrows():

    family = row[
        "family"
    ]

    family_weight = float(
        row[
            "family_weight"
        ]
    )

    img = family_score_images[
        family
    ]

    valid = (
        img.mask()
        .reduce(
            ee.Reducer.min()
        )
        .gt(0)
    )

    overall_sum = (
        overall_sum
        .add(
            img
            .unmask(0)
            .multiply(
                family_weight
            )
        )
    )

    overall_weight = (
        overall_weight
        .add(
            valid
            .toFloat()
            .multiply(
                family_weight
            )
        )
    )


context_modifier = (
    overall_sum
    .divide(
        overall_weight
        .max(1e-6)
    )
    .updateMask(
        overall_weight.gte(0.75)
    )
    .clamp(
        -1,
        1
    )
    .rename(
        "CONTEXT_MODIFIER_SCORE"
    )
)

context_coverage = (
    overall_weight
    .rename(
        "CONTEXT_MODIFIER_COVERAGE"
    )
)


# ------------------------------------------------------------
# F05 integrated score
# ------------------------------------------------------------

FUSION_AMPLITUDE = 5

integrated_score = (
    core_risk
    .add(
        context_modifier
        .unmask(0)
        .multiply(
            FUSION_AMPLITUDE
        )
    )
    .clamp(
        0,
        100
    )
    .rename(
        "INTEGRATED_SCORE_0_100"
    )
)


# ------------------------------------------------------------
# Core evidence
# ------------------------------------------------------------

optical_valid = (
    core_img
    .select(
        "OPTICAL_VALID_PERIOD_COUNT"
    )
)

alert_mask = (
    core_img
    .select(
        "ALERT_MASK"
    )
    .round()
)

max_period_class = (
    core_img
    .select(
        "MAX_PERIOD_CLASS"
    )
)

max_consecutive = (
    core_img
    .select(
        "MAX_CONSECUTIVE_ALERT"
    )
)

optical_alert_total = (
    core_img
    .select(
        "GREENNESS_ALERT_PERIODS"
    )
    .add(
        core_img.select(
            "RED_EDGE_ALERT_PERIODS"
        )
    )
    .add(
        core_img.select(
            "MOISTURE_ALERT_PERIODS"
        )
    )
)

has_optical_support = (
    optical_valid.gt(0)
    .And(
        optical_alert_total.gt(0)
    )
)


# Preserve 30D raw -> final post-processing decisions.

postprocess_lock = (
    core_raw
    .neq(
        core_class
    )
)

unlocked = (
    postprocess_lock
    .Not()
)


# ------------------------------------------------------------
# Frozen AOI01 thresholds
# ------------------------------------------------------------

T12 = 12
T23 = 22
T34 = 32


modifier_available = (
    context_modifier
    .mask()
    .reduce(
        ee.Reducer.min()
    )
    .gt(0)
)

modifier_pos = (
    context_modifier
    .unmask(0)
    .gt(0)
)

modifier_neg = (
    context_modifier
    .unmask(0)
    .lt(0)
)


# ------------------------------------------------------------
# Evidence-gated transitions
# ------------------------------------------------------------

promote_1_2 = (
    core_class.eq(1)
    .And(unlocked)
    .And(modifier_available)
    .And(modifier_pos)
    .And(
        integrated_score.gte(T12)
    )
    .And(
        alert_mask.eq(1)
    )
)

promote_2_3 = (
    core_class.eq(2)
    .And(unlocked)
    .And(modifier_available)
    .And(modifier_pos)
    .And(
        integrated_score.gte(T23)
    )
    .And(
        has_optical_support
    )
    .And(
        max_period_class.gte(3)
    )
    .And(
        max_consecutive.gte(2)
    )
)

promote_3_4 = (
    core_class.eq(3)
    .And(unlocked)
    .And(modifier_available)
    .And(modifier_pos)
    .And(
        integrated_score.gte(T34)
    )
    .And(
        has_optical_support
    )
    .And(
        max_period_class.gte(3)
    )
    .And(
        max_consecutive.gte(3)
    )
)

demote_2_1 = (
    core_class.eq(2)
    .And(unlocked)
    .And(modifier_available)
    .And(modifier_neg)
    .And(
        integrated_score.lt(T12)
    )
    .And(
        alert_mask.eq(0)
    )
)

demote_3_2 = (
    core_class.eq(3)
    .And(unlocked)
    .And(modifier_available)
    .And(modifier_neg)
    .And(
        integrated_score.lt(T23)
    )
    .And(
        max_consecutive.lte(1)
    )
)

demote_4_3 = (
    core_class.eq(4)
    .And(unlocked)
    .And(modifier_available)
    .And(modifier_neg)
    .And(
        integrated_score.lt(T34)
    )
    .And(
        max_consecutive.lte(2)
    )
)


# ------------------------------------------------------------
# Maximum +/-1 class refinement
# ------------------------------------------------------------

integrated_class = (
    core_class
    .where(
        promote_1_2,
        2
    )
    .where(
        promote_2_3,
        3
    )
    .where(
        promote_3_4,
        4
    )
    .where(
        demote_2_1,
        1
    )
    .where(
        demote_3_2,
        2
    )
    .where(
        demote_4_3,
        3
    )
    .rename(
        "INTEGRATED_CLASS"
    )
    .toInt16()
)

class_delta = (
    integrated_class
    .subtract(
        core_class
    )
    .rename(
        "CLASS_DELTA"
    )
    .toInt16()
)


# ------------------------------------------------------------
# Reconstructed seven-band product
# ------------------------------------------------------------

integrated_img = (
    core_class
    .rename(
        "CORE_CLASS"
    )
    .addBands(
        core_risk.rename(
            "CORE_RISK_SCORE_0_100"
        )
    )
    .addBands(
        context_modifier
    )
    .addBands(
        context_coverage
    )
    .addBands(
        integrated_score
    )
    .addBands(
        integrated_class
    )
    .addBands(
        class_delta
    )
    .updateMask(
        valid_mask
    )
)

print(
    "Reconstructed bands:"
)

print(
    integrated_img
    .bandNames()
    .getInfo()
)

## 30G Canonical Reproducibility Check

The reconstructed frozen product is compared against the
permanent canonical Earth Engine asset.

This is a reproducibility / implementation check.

It is **not field accuracy**.

In [ ]:
canonical_img = ee.Image(
    ASSETS["CANONICAL"]
)

canonical_proj_info = (
    canonical_img
    .select(
        "INTEGRATED_CLASS"
    )
    .projection()
    .getInfo()
)

canonical_crs = (
    canonical_proj_info[
        "crs"
    ]
)

canonical_transform = (
    canonical_proj_info[
        "transform"
    ]
)


reconstructed_class_aligned = (
    integrated_img
    .select(
        "INTEGRATED_CLASS"
    )
    .reproject(
        crs=canonical_crs,
        crsTransform=canonical_transform
    )
)


class_mismatch = (
    reconstructed_class_aligned
    .neq(
        canonical_img.select(
            "INTEGRATED_CLASS"
        )
    )
    .rename(
        "MISMATCH"
    )
)


mismatch_area_ha = (
    ee.Image.pixelArea()
    .divide(10000)
    .rename(
        "AREA_HA"
    )
    .updateMask(
        class_mismatch
    )
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=canonical_img.geometry(),
        crs=canonical_crs,
        crsTransform=canonical_transform,
        maxPixels=1e13,
        tileScale=4
    )
    .get(
        "AREA_HA"
    )
    .getInfo()
)

if mismatch_area_ha is None:
    mismatch_area_ha = 0.0


band_names = (
    canonical_img
    .bandNames()
    .getInfo()
)


expected_bands = [
    "CORE_CLASS",
    "CORE_RISK_SCORE_0_100",
    "CONTEXT_MODIFIER_SCORE",
    "CONTEXT_MODIFIER_COVERAGE",
    "INTEGRATED_SCORE_0_100",
    "INTEGRATED_CLASS",
    "CLASS_DELTA",
]


checks = {

    "seven_canonical_bands":
        band_names == expected_bands,

    "reconstructed_class_mismatch_zero":
        float(
            mismatch_area_ha
        ) == 0.0,

    "frozen_fusion_amplitude":
        FUSION_AMPLITUDE == 5,

    "frozen_thresholds":
        (
            T12 == 12
            and T23 == 22
            and T34 == 32
        ),
}


print(
    "Canonical class mismatch area:",
    f"{float(mismatch_area_ha):.6f} ha"
)

print()

for name, result in checks.items():

    print(
        f"{name:40s}:",
        "PASS"
        if result
        else "FAIL"
    )


if not all(
    checks.values()
):
    raise RuntimeError(
        "Canonical reproducibility check failed."
    )

print(
    "\n30G canonical reproducibility : PASS"
)

# 30H | Analytical Freeze

30H records the final canonical statistics, legend,
band dictionary, final map, product summary and freeze
manifest.

No further AOI01 model tuning is performed unless new,
independent validation evidence justifies revision.

In [ ]:
CHECKPOINT_30H = (
    PROJECT_ROOT /
    "checkpoints_30H"
)

required_30h = [
    "30H_FINAL_CLASS_AREA.csv",
    "30H_FINAL_TRANSITION_AREA.csv",
    "30H_PRODUCT_SUMMARY.json",
    "30H_LEGEND.json",
    "30H_BAND_DICTIONARY.json",
    "30H_FREEZE_MANIFEST.json",
    "30H_FINAL_MAP.png",
    "30H_PRODUCT_SUMMARY.md",
]

missing_30h = [
    name
    for name in required_30h
    if not (
        CHECKPOINT_30H /
        name
    ).exists()
]

if missing_30h:
    raise FileNotFoundError(
        f"Missing 30H files: {missing_30h}"
    )


class_area_df = pd.read_csv(
    CHECKPOINT_30H /
    "30H_FINAL_CLASS_AREA.csv"
)

transition_df = pd.read_csv(
    CHECKPOINT_30H /
    "30H_FINAL_TRANSITION_AREA.csv"
)

display(class_area_df)

print(
    "30H required files : PASS"
)

# 30I | Web Delivery

The web-delivery contract separates the cloud analytical
source of truth from distribution artifacts.

Primary raster delivery:

1. `SIAK_AOI01_O3_CLASS_COG_2025.tif`
2. `SIAK_AOI01_O3_METRICS_COG_2025.tif`

COGs are delivery products.

They do not replace the permanent Earth Engine analytical
asset.

In [ ]:
COG_STATUS_PATH = (
    PACKAGE_ROOT /
    "11_LOGS" /
    "30I_COG_EXPORT_STATUS_SNAPSHOT.json"
)

if COG_STATUS_PATH.exists():

    with open(
        COG_STATUS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        cog_status = json.load(f)

    print(
        json.dumps(
            cog_status,
            indent=2
        )
    )

else:

    print(
        "Historical COG status snapshot "
        "not found."
    )


WEB_DIR = (
    PROJECT_ROOT /
    "WEB_DELIVERY_OUTPUT3"
)

print(
    "\nLocal web-delivery source:",
    WEB_DIR
)

print(
    "Exists:",
    WEB_DIR.exists()
)

# Master Pipeline Gate

Passing this gate means:

- configuration is readable;
- persistent source assets are available;
- frozen 30F parameters exist;
- 30G can be reconstructed;
- reconstructed class product agrees with the canonical asset;
- 30H freeze products are available;
- 30I delivery provenance is recorded.

It does **not** mean independent field accuracy has been
demonstrated.

In [ ]:
master_gate = {

    "configuration":
        CONFIG_PATH.exists(),

    "gee_asset_inventory":
        ASSET_INVENTORY_PATH.exists(),

    "frozen_calibration":
        CALIBRATION_PATH.exists(),

    "frozen_family_weights":
        FAMILY_WEIGHT_PATH.exists(),

    "canonical_asset_7_bands":
        band_names == expected_bands,

    "canonical_class_reproducibility":
        float(
            mismatch_area_ha
        ) == 0.0,

    "30h_freeze":
        len(
            missing_30h
        ) == 0,
}


print(
    "=" * 90
)

print(
    "MASTER_PIPELINE_OUTPUT3 FINAL GATE"
)

print(
    "=" * 90
)


for name, result in master_gate.items():

    print(
        f"{name:40s}:",
        "PASS"
        if result
        else "FAIL"
    )


if all(
    master_gate.values()
):

    print("""
MASTER PIPELINE RESULT: PASS

AOI01 Output 3 is reproducible from the frozen
Earth Engine source assets and frozen Python
calibration parameters.

This is a reproducibility / implementation result,
not an independent field-accuracy claim.
""")

else:

    raise RuntimeError(
        "Master pipeline gate failed."
    )

# Historical Working Notebook

The complete experimental / development notebook used during
Output 3 development is preserved separately as:

`OUTPUT3_WORKING_NOTEBOOK_SNAPSHOT_20260813_114651.ipynb`

A cell-level index is preserved as:

`OUTPUT3_WORKING_NOTEBOOK_CELL_INDEX.csv`

This separation keeps the official master notebook clean and
safe to rerun while retaining the full research/development
history for audit and handover.